# Design Freeze v2——梁上真實分佈載重版

**取代 `design_freeze.json`(v1)**:v1 用的是重力載重集中在柱頂
節點這個簡化模型算出的配筋,被一張網路圖揭露這個局限後,`Stage
4/5` 已經用梁上真實分佈載重全部重做過(`stage4_5_udl_rebuild.
ipynb`)。這份 `notebook` 重新執行同一套計算,組裝成更新版的
`design_freeze.json`。

In [1]:

import json, os
try:
    import openseespy.opensees as ops
except ImportError:
    import sys
    !{sys.executable} -m pip install -q openseespy
    import openseespy.opensees as ops
if not os.path.exists('rc_design.py'):
    !wget -q -O rc_design.py https://raw.githubusercontent.com/zhixiu0223/taiwan-seismic-code-calc/main/rc_design.py

from rc_design import design_column_PM, design_rebar

def Mn_at_Pu(curve, Pu_target):
    sc = sorted(curve, key=lambda p: p['Pn'])
    for i in range(len(sc)-1):
        if sc[i]['Pn'] <= Pu_target <= sc[i+1]['Pn']:
            t = (Pu_target-sc[i]['Pn'])/(sc[i+1]['Pn']-sc[i]['Pn'])
            return sc[i]['Mn'] + t*(sc[i+1]['Mn']-sc[i]['Mn'])
    return None

def compute_Mpr(As_cm2, b_cm, d_cm, fc=280.0, fy=4200.0):
    fy_pr = 1.25*fy
    a = As_cm2*fy_pr/(0.85*fc*b_cm)
    return (As_cm2*fy_pr*(d_cm - a/2)) * 9.80665e-5

def design_column_rebar(b_cm, h_cm, Pu_kN, Mu_kNm, fc=280.0, fy=4200.0, Es=2.0e6,
                          cover_to_center=4.0, rho_candidates=None):
    if rho_candidates is None:
        rho_candidates = [0.010, 0.012, 0.014, 0.016, 0.018, 0.020, 0.025, 0.030, 0.040]
    for rho in rho_candidates:
        As_bar = rho*b_cm*h_cm/8
        As_layers = [3*As_bar, 2*As_bar, 3*As_bar]
        r = design_column_PM(b_cm=b_cm, h_cm=h_cm, As_per_layer_cm2=As_layers,
                               cover_to_center_cm=cover_to_center, n_layers=3,
                               fc=fc, fy=fy, Es=Es, Pu_kN=Pu_kN, Mu_kNm=Mu_kNm)
        if r['within_envelope']:
            return dict(rho=rho, As_total=8*As_bar, r=r, Mn=Mn_at_Pu(r['curve'], Pu_kN))
    return None

def design_column_confinement(h_col_cm, Ln_cm, db_long_cm, fc=280.0, fyt=4200.0, cover_cm=4.0):
    lo = max(h_col_cm, Ln_cm/6, 45.0)
    bc = h_col_cm - 2*cover_cm
    so = min(max(10.0 + (35.0-bc)/3.0, 10.0), 15.0)
    s_max = min(h_col_cm/4, 6*db_long_cm, so)
    Ag = h_col_cm**2
    Ach = bc**2
    Ash_over_s_req = max(0.3*bc*(fc/fyt)*(Ag/Ach-1), 0.09*bc*fc/fyt)
    Ash_2leg = 2*0.7133
    return dict(lo=lo, s_max=s_max, s_final=min(Ash_2leg/Ash_over_s_req, s_max))

print("核心函式定義完成")


核心函式定義完成


## 重新執行線彈性分析(梁上真實均布載重)

In [2]:

L_bay = 6.0; h1 = 3.5; h2 = 3.5
h_col = 0.40; b_beam, h_beam = 0.30, 0.50
E_rc = 2.463e7
Ic = 0.7*(h_col**4/12); Ib = 0.35*(b_beam*h_beam**3/12)
Ac = h_col**2; Ab = b_beam*h_beam
F1_Y, F2_Y = 9.938, 15.900
w_udl = 49.20

def build_with_midspan():
    ops.wipe(); ops.model('basic', '-ndm', 2, '-ndf', 3)
    ops.node(1, 0.0, 0.0);   ops.node(2, L_bay, 0.0)
    ops.node(3, 0.0, h1);    ops.node(4, L_bay, h1)
    ops.node(5, 0.0, h1+h2); ops.node(6, L_bay, h1+h2)
    ops.node(7, L_bay/2, h1)
    ops.node(8, L_bay/2, h1+h2)
    ops.fix(1,1,1,1); ops.fix(2,1,1,1)
    ops.geomTransf('Linear', 1)
    ops.element('elasticBeamColumn', 1, 1, 3, Ac, E_rc, Ic, 1)
    ops.element('elasticBeamColumn', 2, 2, 4, Ac, E_rc, Ic, 1)
    ops.element('elasticBeamColumn', 3, 3, 5, Ac, E_rc, Ic, 1)
    ops.element('elasticBeamColumn', 4, 4, 6, Ac, E_rc, Ic, 1)
    ops.element('elasticBeamColumn', 5, 3, 7, Ab, E_rc, Ib, 1)
    ops.element('elasticBeamColumn', 9, 7, 4, Ab, E_rc, Ib, 1)
    ops.element('elasticBeamColumn', 6, 5, 8, Ab, E_rc, Ib, 1)
    ops.element('elasticBeamColumn', 10, 8, 6, Ab, E_rc, Ib, 1)

def run_gravity():
    build_with_midspan()
    ops.timeSeries('Linear', 1); ops.pattern('Plain', 1, 1)
    ops.eleLoad('-ele', 5, 9, '-type', '-beamUniform', -w_udl)
    ops.eleLoad('-ele', 6, 10, '-type', '-beamUniform', -w_udl)
    ops.system('BandGeneral'); ops.numberer('Plain'); ops.constraints('Plain')
    ops.algorithm('Linear'); ops.integrator('LoadControl', 1.0); ops.analysis('Static')
    ops.analyze(1)
    return (ops.eleForce(1), ops.eleForce(2), ops.eleForce(5), ops.eleForce(9),
            ops.eleForce(3), ops.eleForce(4), ops.eleForce(6), ops.eleForce(10))

def run_lateral():
    build_with_midspan()
    ops.timeSeries('Linear', 1); ops.pattern('Plain', 1, 1)
    ops.load(3, F1_Y, 0.0, 0.0); ops.load(5, F2_Y, 0.0, 0.0)
    ops.system('BandGeneral'); ops.numberer('Plain'); ops.constraints('Plain')
    ops.algorithm('Linear'); ops.integrator('LoadControl', 1.0); ops.analysis('Static')
    ops.analyze(1)
    return (ops.eleForce(1), ops.eleForce(2), ops.eleForce(5), ops.eleForce(9),
            ops.eleForce(3), ops.eleForce(4), ops.eleForce(6), ops.eleForce(10))

g = run_gravity(); l = run_lateral()
labels = ['1F左柱', '1F右柱', '1F樑左半', '1F樑右半', '2F左柱', '2F右柱', '屋頂樑左半', '屋頂樑右半']
governing = {}
for gf, lf, label in zip(g, l, labels):
    for end, idx in [('i',2),('j',5)]:
        Mg, Ml = gf[idx], lf[idx]
        governing[(label,end)] = max(abs(Mg+Ml), abs(Mg-Ml))

Mu_1F_col = max(governing[('1F左柱','i')], governing[('1F左柱','j')], governing[('1F右柱','i')], governing[('1F右柱','j')])
Mu_2F_col = max(governing[('2F左柱','i')], governing[('2F左柱','j')], governing[('2F右柱','i')], governing[('2F右柱','j')])
Mu_1F_beam_neg = governing[('1F樑左半','i')]
Mu_1F_beam_pos = governing[('1F樑左半','j')]
Mu_roof_beam_neg = governing[('屋頂樑左半','i')]
Mu_roof_beam_pos = governing[('屋頂樑左半','j')]
print(f"1F柱Mu={Mu_1F_col:.2f}, 2F柱Mu={Mu_2F_col:.2f}")
print(f"1F樑Mu-={Mu_1F_beam_neg:.2f}, Mu+={Mu_1F_beam_pos:.2f}")
print(f"屋頂樑Mu-={Mu_roof_beam_neg:.2f}, Mu+={Mu_roof_beam_pos:.2f}")


1F柱Mu=57.17, 2F柱Mu=141.95
1F樑Mu-=162.52, Mu+=83.07
屋頂樑Mu-=141.95, Mu+=96.24


## 重新設計配筋跟全部 `Stage 5` 檢核

In [3]:

Pu_1F_col, Pu_2F_col = 308.85, 153.19
result_1F_col = design_column_rebar(40.0, 40.0, Pu_1F_col, Mu_1F_col)
result_2F_col = design_column_rebar(40.0, 40.0, Pu_2F_col, Mu_2F_col)
r_1F_neg = design_rebar(Mu_1F_beam_neg, 30.0, 50.0, cover=4.0)
r_1F_pos = design_rebar(Mu_1F_beam_pos, 30.0, 50.0, cover=4.0)
r_roof_neg = design_rebar(Mu_roof_beam_neg, 30.0, 50.0, cover=4.0)
r_roof_pos = design_rebar(Mu_roof_beam_pos, 30.0, 50.0, cover=4.0)

Mn_1F_neg = r_1F_neg['phiMn_provided']/0.9
Mn_1F_pos = r_1F_pos['phiMn_provided']/0.9
Mn_roof_neg = r_roof_neg['phiMn_provided']/0.9
Mn_roof_pos = r_roof_pos['phiMn_provided']/0.9
ratio_posneg_1F = Mn_1F_pos/Mn_1F_neg
ratio_posneg_roof = Mn_roof_pos/Mn_roof_neg

sum_Mnc_1F = result_1F_col['Mn'] + result_2F_col['Mn']
scwb_1F_ratio = sum_Mnc_1F/Mn_1F_neg
limit_exempt = (1600.0*280.0/10)*9.80665e-3
roof_scwb_exempt = Pu_2F_col < limit_exempt

Mpr_1F_neg = compute_Mpr(r_1F_neg['As_provided'], 30.0, r_1F_neg['d'])
Mpr_1F_pos = compute_Mpr(r_1F_pos['As_provided'], 30.0, r_1F_pos['d'])
Mpr_roof_neg = compute_Mpr(r_roof_neg['As_provided'], 30.0, r_roof_neg['d'])
Mpr_roof_pos = compute_Mpr(r_roof_pos['As_provided'], 30.0, r_roof_pos['d'])
Ln_beam = L_bay - h_col
Ve_hinge_1F_beam = (Mpr_1F_neg + Mpr_1F_pos)/Ln_beam
Ve_hinge_roof_beam = (Mpr_roof_neg + Mpr_roof_pos)/Ln_beam

def beam_stirrup_spacing(Ve_hinge, d_beam, db_long=1.9):
    Vn_req = Ve_hinge/0.75
    Vs_req_kgf = Vn_req/9.80665e-3
    s_from_Vs = (2*0.7133*4200.0*d_beam)/Vs_req_kgf
    return min(s_from_Vs, min(d_beam/4, 6*db_long, 15.0))

s_1F_beam = beam_stirrup_spacing(Ve_hinge_1F_beam, r_1F_neg['d'])
s_roof_beam = beam_stirrup_spacing(Ve_hinge_roof_beam, r_roof_neg['d'])

conf_1F = design_column_confinement(40.0, (h1-0.5)*100, 1.6)
conf_2F = design_column_confinement(40.0, (h2-0.5)*100, 1.8)

fy_pr_col = 1.25*4200.0
col_shear = {}
for label, Pu, rho_col, Mpr_beam_adj in [('1F_column', Pu_1F_col, result_1F_col['rho'], Mpr_1F_neg),
                                            ('2F_column', Pu_2F_col, result_2F_col['rho'], Mpr_roof_neg)]:
    As_bar = rho_col*40.0*40.0/8
    As_layers = [3*As_bar, 2*As_bar, 3*As_bar]
    r_pr = design_column_PM(b_cm=40.0, h_cm=40.0, As_per_layer_cm2=As_layers,
                              cover_to_center_cm=4.0, n_layers=3,
                              fc=280.0, fy=fy_pr_col, Es=2.0e6, Pu_kN=Pu, Mu_kNm=1.0)
    Mpr_col = Mn_at_Pu(r_pr['curve'], Pu)
    Ve_gov = min(2*Mpr_col/3.0, Mpr_beam_adj/3.0)
    col_shear[label] = Ve_gov

def joint_shear_check(As_top, Mpr_beam, h_avg):
    T1_kN = (As_top*5250.0)*9.80665e-3
    Vcol_kN = Mpr_beam/h_avg
    Vj_kN = T1_kN - Vcol_kN
    fc_psi = 280.0*14.223
    Aj_in2 = 1600.0/6.4516
    Vn_kN = (10*(fc_psi**0.5)*Aj_in2/1000)*4.448
    phiVn_kN = 0.85*Vn_kN
    return dict(Vj_kN=Vj_kN, phiVn_kN=phiVn_kN, ratio=phiVn_kN/Vj_kN)

j1 = joint_shear_check(r_1F_neg['As_provided'], Mpr_1F_neg, (h1+h2)/2)
j2 = joint_shear_check(r_roof_neg['As_provided'], Mpr_roof_neg, h2)

print("全部重新計算完成")


全部重新計算完成


## 組裝 `design_freeze.json`(`v2`)

In [4]:

design_freeze_v2 = {
    "status": "FROZEN",
    "version": 2,
    "supersedes": "design_freeze.json (v1, deprecated: used concentrated gravity load at column-top nodes, not realistic distributed load on beam)",
    "source_repo": "taiwan-seismic-code-calc",
    "change_log": "v2改用梁上真實分佈載重(w=49.2kN/m, 換算自原本集中力, 總重力量級不變只改變傳遞路徑), 取代v1的簡化模型——由一張網路RC構架裂縫示意圖揭露v1無法重現梁跨中正彎矩/梁端負彎矩的典型撓曲行為而觸發這次重建",
    "geometry": {"L_bay_m": L_bay, "h1_m": h1, "h2_m": h2,
                  "col_section_cm": "40x40", "beam_section_cm": "30x50"},
    "materials": {"fc_kgf_cm2": 280.0, "fy_kgf_cm2": 4200.0, "E_kPa": E_rc,
                   "stiffness_reduction": "beam 0.35Ig, column 0.7Ig"},
    "demands": {
        "1F_column": {"Pu_kN": Pu_1F_col, "Mu_kNm": round(Mu_1F_col,2)},
        "2F_column": {"Pu_kN": Pu_2F_col, "Mu_kNm": round(Mu_2F_col,2)},
        "1F_beam": {"Mu_neg_kNm": round(Mu_1F_beam_neg,2), "Mu_pos_kNm": round(Mu_1F_beam_pos,2),
                     "w_udl_kN_m": w_udl, "note": "正彎矩(跨中)/負彎矩(端點)首次分開求解, v1沒有這項資訊"},
        "roof_beam": {"Mu_neg_kNm": round(Mu_roof_beam_neg,2), "Mu_pos_kNm": round(Mu_roof_beam_pos,2)},
    },
    "members": {
        "1F_column": {"section_cm": "40x40", "rho": result_1F_col['rho'],
                       "As_total_cm2": round(result_1F_col['As_total'],2),
                       "governing_factor": "ACI 318 minimum reinforcement ratio",
                       "Mn_kNm": round(result_1F_col['Mn'],2),
                       "utilization": round(result_1F_col['r']['utilization'],4),
                       "confinement": {"lo_cm": conf_1F['lo'], "tie_spacing_cm": round(conf_1F['s_final'],2)},
                       "capacity_design_shear_kN": round(col_shear['1F_column'],2)},
        "2F_column": {"section_cm": "40x40", "rho": result_2F_col['rho'],
                       "As_total_cm2": round(result_2F_col['As_total'],2),
                       "governing_factor": "STRENGTH DEMAND (not minimum ratio) -- first time this governs in this pipeline",
                       "Mn_kNm": round(result_2F_col['Mn'],2),
                       "utilization": round(result_2F_col['r']['utilization'],4),
                       "confinement": {"lo_cm": conf_2F['lo'], "tie_spacing_cm": round(conf_2F['s_final'],2)},
                       "capacity_design_shear_kN": round(col_shear['2F_column'],2)},
        "1F_beam": {"section_cm": "30x50",
                     "top_As_cm2": round(r_1F_neg['As_provided'],2), "top_bar": f"{r_1F_neg['bar_size']}x{r_1F_neg['n_bars']}",
                     "bottom_As_cm2": round(r_1F_pos['As_provided'],2), "bottom_bar": f"{r_1F_pos['bar_size']}x{r_1F_pos['n_bars']}",
                     "posneg_ratio": round(ratio_posneg_1F,3),
                     "posneg_ratio_note": "CLOSE TO 0.5 LIMIT, not comfortably passing" if ratio_posneg_1F < 0.55 else "OK",
                     "Mpr_top_kNm": round(Mpr_1F_neg,2), "Mpr_bottom_kNm": round(Mpr_1F_pos,2),
                     "capacity_design_shear_kN": round(Ve_hinge_1F_beam,2),
                     "stirrup_spacing_hinge_zone_cm": round(s_1F_beam,2)},
        "roof_beam": {"section_cm": "30x50",
                       "top_As_cm2": round(r_roof_neg['As_provided'],2), "top_bar": f"{r_roof_neg['bar_size']}x{r_roof_neg['n_bars']}",
                       "bottom_As_cm2": round(r_roof_pos['As_provided'],2), "bottom_bar": f"{r_roof_pos['bar_size']}x{r_roof_pos['n_bars']}",
                       "posneg_ratio": round(ratio_posneg_roof,3),
                       "Mpr_top_kNm": round(Mpr_roof_neg,2), "Mpr_bottom_kNm": round(Mpr_roof_pos,2),
                       "capacity_design_shear_kN": round(Ve_hinge_roof_beam,2),
                       "stirrup_spacing_hinge_zone_cm": round(s_roof_beam,2)},
    },
    "scwb": {
        "1F_joint": {"sum_Mnc_kNm": round(sum_Mnc_1F,2), "ratio": round(scwb_1F_ratio,2),
                      "status": "PASS" if scwb_1F_ratio>=1.2 else "FAIL",
                      "note": "ratio dropped from 2.99(v1) to this value because beam reinforcement increased substantially"},
        "roof_joint": {"status": "EXEMPT", "reason": "column discontinuous above + Pu<Ag*fc'/10",
                        "code_ref": "ACI 318-19 18.7.3.1 exception"},
    },
    "joint_shear": {
        "1F_joint": {"Vj_kN": round(j1['Vj_kN'],2), "phiVn_kN": round(j1['phiVn_kN'],2),
                      "ratio": round(j1['ratio'],2), "status": "PASS" if j1['ratio']>=1.0 else "FAIL",
                      "note": "margin dropped from 2.28(v1) to this value, tightest check in this design"},
        "roof_joint": {"Vj_kN": round(j2['Vj_kN'],2), "phiVn_kN": round(j2['phiVn_kN'],2),
                        "ratio": round(j2['ratio'],2), "status": "PASS" if j2['ratio']>=1.0 else "FAIL"},
    },
    "assumptions": [
        "Stage 2/3 canonical elastic model: 單榀2D substructure, 樓板內力分配(多榀構架之間)不在這個模型求解範圍內",
        "梁上均布重力載重w=49.2kN/m, 換算自原本集中力總量(147.6*2/6.0), 總重力量級不變只改變傳遞路徑",
        "有效勁度折減(0.35Ig/0.7Ig)是這個分析目的下宣告的建模假設, 不是唯一或絕對正確的RC真實勁度",
        "柱斷面(40x40cm)、梁斷面(30x50cm)是Case-03.5工程師手動選定的初步試設尺寸, 不是本次Stage 4的尺寸最佳化結果",
        "未涵蓋範圍(不在這次凍結內): 錨定/搭接長度、柱主筋沿全長排筋細節、接頭橫向鋼筋排列細節",
        "需要特別留意: 1F樑正負彎矩比值(0.521)跟1F接頭剪力餘裕(1.13倍)都明顯貼近規範門檻, 不是輕鬆通過",
    ],
    "code_references": ["ACI 318-19 Chapter 18 (Special moment frames)", "ACI 318-19 22.4 (PM interaction)",
                          "ACI 318-19 9.6.1.2/10.6.1.1 (minimum reinforcement)", "ACI 318-19 18.7.3.1 (roof joint exemption)"],
}

with open('design_freeze.json', 'w', encoding='utf-8') as f:
    json.dump(design_freeze_v2, f, indent=2, ensure_ascii=False)
print("design_freeze.json(v2) 產生完成")


design_freeze.json(v2) 產生完成


## 驗證:跟 `stage4_5_udl_rebuild.ipynb` 的數字逐項比對

In [5]:

checks = [
    (result_1F_col['Mn'], 159.97, "1F柱Mn"),
    (result_2F_col['Mn'], 157.45, "2F柱Mn"),
    (result_2F_col['rho'], 0.012, "2F柱rho"),
    (sum_Mnc_1F, 317.42, "1F接頭sum(Mnc)"),
    (ratio_posneg_1F, 0.521, "1F樑Mn+/Mn-比值"),
    (j1['ratio'], 1.13, "1F接頭剪力餘裕"),
]
all_pass = True
for computed, reference, label in checks:
    diff = abs(computed-reference)
    ok = diff < max(0.5, abs(reference)*0.02)
    all_pass = all_pass and ok
    print(f"{label}: 重新計算={computed:.3f}, 原notebook={reference}, 差異={diff:.4f} {'OK' if ok else 'MISMATCH'}")

assert all_pass
print("\n[PASS] design_freeze.json(v2)跟stage4_5_udl_rebuild.ipynb的關鍵數字一致")


1F柱Mn: 重新計算=159.973, 原notebook=159.97, 差異=0.0027 OK
2F柱Mn: 重新計算=157.447, 原notebook=157.45, 差異=0.0031 OK
2F柱rho: 重新計算=0.012, 原notebook=0.012, 差異=0.0000 OK
1F接頭sum(Mnc): 重新計算=317.420, 原notebook=317.42, 差異=0.0004 OK
1F樑Mn+/Mn-比值: 重新計算=0.521, 原notebook=0.521, 差異=0.0002 OK
1F接頭剪力餘裕: 重新計算=1.131, 原notebook=1.13, 差異=0.0008 OK

[PASS] design_freeze.json(v2)跟stage4_5_udl_rebuild.ipynb的關鍵數字一致
